# SHIPP Stage 5B — Silver Requests Development
Bronze bootcamp_students.shipp_bronze.lb_requests_history → Silver bootcamp_students.shipp_silver.silver_requests

Bronze CDC history (pinned Delta version)
  → keep insert + update_postimage + delete
  → newest event per request_id   (_pg_lsn DESC, _sort_by DESC)
  → drop keys whose newest event is delete
  → Silver contract + quality gate
  → overwrite Silver
  → reconcile Silver vs Bronze (two independent methods)
  → idempotency check

**Method:** Spark Window Current-State Reconstruction

```text
Bronze request history
    ↓
partition by request_id
    ↓
newest CDC event
    ↓
quality checks
    ↓
silver_requests
```

How to run
Close any AI assistant panel that has auto-approve on.
Run → Clear state and outputs, then Run all. Never run single cells out of order.
Every gate either prints PASS or stops with an AssertionError. A failure means the data is wrong — fix the cause, never loosen the assert.
Expected Silver content: every request whose newest Bronze event is not a delete — including demo-request-001.

This notebook is the only writer of silver_requests.


In [0]:
%run ../common/shipp_silver_lib

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SHIPP — notebook bootstrap (do not run directly)
Every stage notebook starts with:

%run ../common/shipp_silver_lib
This cell finds the repo root, reloads the Python modules under config/ and data_pipeline/ (so an edited module is always picked up, even without Clear state), and exposes their functions with the notebook's spark session already bound.

All logic lives in .py files (data_pipeline/…), covered by tests/unit in CI. Notebooks only read, call those functions, run gates and write.

SHIPP bootstrap loaded from /Workspace/Users/raneem99hamarneh@yahoo.com/Shipp/shipp-marketplace-v3

# Config

In [0]:
CATALOG = "bootcamp_students"

SOURCE_TABLE = f"{CATALOG}.shipp_bronze.lb_requests_history"
TARGET_SCHEMA = f"{CATALOG}.shipp_silver"
TARGET_TABLE = f"{TARGET_SCHEMA}.silver_requests"
BUSINESS_KEY = "request_id"

BRONZE_SCHEMA = "shipp_bronze"
SILVER_SCHEMA = "shipp_silver"

# NOTE: these request statuses follow the spec (Open → MatchesAvailable → ItemSaved → Closed).
# If the status gate fails, the printed 'Distinct statuses' shows what Lakebase really stores:
# update this set to match the Lakebase CHECK constraint — do NOT delete the check.
VALID_STATUSES = ["OPEN", "MATCHES_AVAILABLE", "ITEM_SAVED", "CLOSED"]

REQUIRED_SOURCE_COLUMNS = {
    "request_id",
    "requester_id",
    "request_text",
    "category",
    "location",
    "latitude",
    "longitude",
    "need_by_date",
    "status",
    "updated_at",
    "_pg_change_type",
    "_pg_lsn",
    "_sort_by",
}

EXPECTED_SILVER_COLUMNS = [
    "request_id",
    "requester_id",
    "request_text",
    "category",
    "latitude",
    "longitude",
    "need_by_date",
    "status",
    "updated_at",
    "source_pg_lsn",
    "source_sort_by",
    "source_change_type",
    "silver_processed_at",
]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

print("SOURCE_TABLE =", SOURCE_TABLE)
print("TARGET_TABLE =", TARGET_TABLE)
print("BUSINESS_KEY =", BUSINESS_KEY)

# Load Bronze (pinned) + source Schema contract

In [0]:
bronze_df, bronze_version = read_bronze_pinned(SOURCE_TABLE)

# R1 validation — verify real CDC ordering column datatypes
print("=== REAL BRONZE CDC ORDERING TYPES ===")
print("_pg_lsn type   :", bronze_df.schema["_pg_lsn"].dataType)
print("_sort_by type  :", bronze_df.schema["_sort_by"].dataType)

print("\n=== REAL BRONZE SCHEMA ===")
bronze_df.printSchema()

print("Bronze row count:", bronze_df.count())

#  Delete-aware current-state reconstruction

In [0]:
missing_columns = REQUIRED_SOURCE_COLUMNS - set(bronze_df.columns)

assert not missing_columns, (
    f"Bronze schema contract failed. Missing: {sorted(missing_columns)}"
)

print("PASS — Bronze source schema contract.")

display(
    bronze_df
    .groupBy("_pg_change_type")
    .count()
    .orderBy("_pg_change_type")
)

In [0]:
latest_events_df, current_rows_df, deleted_keys_df = reconstruct_current_state(bronze_df, BUSINESS_KEY)

print("Keys with any event:        ", latest_events_df.count())
print("Newest event = delete (out):", deleted_keys_df.count())
print("Current rows for Silver:    ", current_rows_df.count())

display(
    latest_events_df
    .select(BUSINESS_KEY, "_pg_change_type", "_pg_lsn", "_sort_by", "status")
    .orderBy(BUSINESS_KEY)
)

In [0]:
print("=== CHECK: duplicate CDC ordering keys — REQUESTS ===")

request_ties = (
    bronze_df
    .groupBy(
        "request_id",
        "_pg_lsn",
        "_sort_by",
    )
    .count()
    .filter(F.col("count") > 1)
)

request_tie_count = request_ties.count()

print("Duplicate (request_id, _pg_lsn, _sort_by) groups:", request_tie_count)

display(
    request_ties.orderBy(
        "request_id",
        "_pg_lsn",
        "_sort_by",
    )
)

assert request_tie_count == 0, (
    "FOUND real duplicate CDC ordering keys in Requests."
)

print("PASS — Requests CDC ordering keys are unique.")

# Silver contract

In [0]:
silver_df = current_rows_df.select(
    # Business columns
    F.col("request_id"),
    F.col("requester_id"),
    F.col("request_text"),
    F.col("category"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("need_by_date"),
    F.col("status"),
    F.col("updated_at"),

    # Traceability columns
    F.col("_pg_lsn").alias("source_pg_lsn"),
    F.col("_sort_by").alias("source_sort_by"),
    F.col("_pg_change_type").alias("source_change_type"),

    # Silver processing metadata
    F.current_timestamp().alias("silver_processed_at"),
)

print("Silver candidate rows:", silver_df.count())
display(silver_df.orderBy(BUSINESS_KEY))

# Pre-write quality gate

In [0]:
row_count = silver_df.count()

print("Distinct statuses:", sorted(r["status"] for r in silver_df.select("status").distinct().collect()))

violations = {
    "duplicate request_id": silver_df.groupBy(BUSINESS_KEY).count().filter(F.col("count") > 1).count(),
    "null request_id": silver_df.filter(F.col("request_id").isNull()).count(),
    "null requester_id": silver_df.filter(F.col("requester_id").isNull()).count(),
    "null request_text": silver_df.filter(F.col("request_text").isNull()).count(),
    "null category": silver_df.filter(F.col("category").isNull()).count(),
    "null need_by_date": silver_df.filter(F.col("need_by_date").isNull()).count(),
    "null status": silver_df.filter(F.col("status").isNull()).count(),
    "latitude out of range": silver_df.filter(
        F.col("latitude").isNotNull() & ~F.col("latitude").between(-90, 90)
    ).count(),
    "longitude out of range": silver_df.filter(
        F.col("longitude").isNotNull() & ~F.col("longitude").between(-180, 180)
    ).count(),
    "invalid status": silver_df.filter(~F.col("status").isin(VALID_STATUSES)).count(),

    "delete event in Silver": silver_df.filter(
        ~F.col("source_change_type").isin(SILVER_SOURCE_CHANGE_TYPES)
    ).count(),
}

print("Silver row count:", row_count)
for name, count in violations.items():
    print(f"  {name:<40} {count}")

failed = {name: count for name, count in violations.items() if count}
assert row_count > 0, "QUALITY FAILED: Silver would contain zero rows."
assert not failed, f"QUALITY FAILED: {failed}"
print("PASS — pre-write quality gate.")


# ## — Write Silver + exact schema check

In [0]:
written_row_count = write_silver(silver_df, TARGET_TABLE)
assert written_row_count == row_count, (
    f"WRITE VALIDATION FAILED: wrote {written_row_count}, validated {row_count}"
)
assert_exact_schema(TARGET_TABLE, EXPECTED_SILVER_COLUMNS)

# Step 7 — Reconcile persisted Silver against Bronze

In [0]:
assert_silver_matches_bronze(
    TARGET_TABLE, bronze_df, current_rows_df, deleted_keys_df, BUSINESS_KEY
)

demo_count = spark.table(TARGET_TABLE).filter(F.col(BUSINESS_KEY) == "demo-request-001").count()
print("demo-request-001 rows in Silver:", demo_count)
assert demo_count == 1, "FAILED: demo record must appear exactly once in Silver."
print("PASS — demo record present exactly once.")

# Idempotency (rewrite the same state, compare logically)

In [0]:
run1 = logical_snapshot(TARGET_TABLE, BUSINESS_KEY)
write_silver(silver_df, TARGET_TABLE)
run2 = logical_snapshot(TARGET_TABLE, BUSINESS_KEY)

print("Run 1 rows:", len(run1), "| Run 2 rows:", len(run2))
assert run1 == run2, "IDEMPOTENCY FAILED: logical Silver contents changed on rewrite."
print("PASS — Silver rewrite is logically idempotent.")

## Step 9 — Exit summary

Returns a JSON summary to a caller (the delete test runs this notebook with `dbutils.notebook.run`).
When you run the notebook yourself you will see *Notebook exited* — that is the expected end.

In [0]:
summary = {
    "table": TARGET_TABLE,
    "rows": spark.table(TARGET_TABLE).count(),
    "bronze_version": bronze_version,
    "status": "PASS",
}
print(summary)
dbutils.notebook.exit(json.dumps(summary))